# User Profile Builder (SBERT Weighted Mean)
`UserVectorBuilder` đã lập trình kỹ thuật nhồi Vector Out-of-Core Batch để xử lý siêu tốc file Interactions 170 triệu dòng khổng lồ.
Không tích lũy RAM, tạo matrix MMap an toàn và sinh file Metadata Tracking `user_history`.

In [1]:
import os
import sys
import numpy as np
from google.colab import drive
drive.mount("/content/drive")

%pip install duckdb torch
sys.path.append(os.path.abspath('/content/drive/My Drive/Thesis/book_recsys'))
from scripts.user_vector_builder import UserVectorBuilder

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1.3.2
thay đổi 1


In [2]:
# Khai báo các đường dẫn hệ thống (Config)

_DATA_DIR = '/content/drive/My Drive/Thesis/book_recsys/data/processed_2'

INTERACTIONS_PQ = f'{_DATA_DIR}/train_main.csv'

version = 'v1'
_MODEL_DIR = f'{_DATA_DIR}/model_{version}'

ITEM_MATRIX = f'{_MODEL_DIR}/cb_sbert_matrix_v2.npy'
ITEM_INDEX = f'{_MODEL_DIR}/cb_sbert_book_index.csv'
DB_HISTORY = f'{_MODEL_DIR}/user_history.db'
USER_INDEX = f'{_MODEL_DIR}/user_index.csv'

OUTPUT_MATRIX = f'{_MODEL_DIR}/cb_sbert_user_matrix_v2_w1.npy'
OUTPUT_INDEX = f'{_MODEL_DIR}/user_index.csv'
OUTPUT_METADATA = f'{_MODEL_DIR}/user_history.db'

CHK_SIZE = 1_000_000 # Cắt 1 triệu dòng mỗi chu kì


In [ ]:
# UserVectorBuilder.build_user_history(
#     interactions_path= INTERACTIONS_PQ,
#     item_index_path = ITEM_INDEX,
#     output_db_path = OUTPUT_METADATA)


Bắt đầu khởi tạo DuckDB Engine cho User History...
Đang thực thi siêu truy vấn xử lý dữ liệu và dội xuống đĩa...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Đang tạo Index cho việc tra cứu...
[Thành công] Toàn bộ lịch sử đã đóng gói bằng DuckDB tại: /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_history.db


In [ ]:
# UserVectorBuilder.export_user_index_csv(db_path = OUTPUT_METADATA, output_csv_path = OUTPUT_INDEX)

Exporting universal User Index to /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_index.csv...
[Success] Universal User Index successfully saved to /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_index.csv


In [3]:
UserVectorBuilder.check_weight()

weight_1


In [5]:
UserVectorBuilder.build_user_matrix_gpu_standalone(
    interactions_path = INTERACTIONS_PQ,
    item_matrix_path=  ITEM_MATRIX,
    user_index_path = USER_INDEX,
    item_index_path = ITEM_INDEX,
    output_matrix_path = OUTPUT_MATRIX,
    )

Using GPU device: Tesla T4
Loading Item Matrix from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/cb_sbert_matrix_v2.npy to GPU...
Loading Mapping Indexes into RAM...
Allocating Matrices on GPU for 743,401 users...
Streaming interactions from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/train_main.csv...
 -> Processed & Accumulated: 178,572,441 interactions
Streaming complete. Executing final normalization phase...
Transferring Normalized Matrix to CPU and saving...
[Success] Standalone GPU Builder finished. Matrix saved to /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/cb_sbert_user_matrix_v2_w1.npy


In [ ]:
import duckdb
import pandas as pd

def verify_user_history_db(db_path: str):
    """
    Connects to the DuckDB user history database in read-only mode
    to verify row counts and data integrity.
    """
    print(f"Connecting to DuckDB database at: {db_path}")

    # Mở kết nối ở chế độ read_only để bảo vệ file
    try:
        conn = duckdb.connect(db_path, read_only=True)
    except Exception as e:
        print(f"Failed to connect. Error: {e}")
        return

    try:
        # 1. Đếm tổng số lượng người dùng (Unique Users)
        total_users_query = "SELECT COUNT(*) FROM user_history"
        total_users = conn.execute(total_users_query).fetchone()[0]
        print(f"Total Unique Users: {total_users:,}")

        # 2. Tính tổng số lượng tương tác đã ghi nhận (Sum of all interactions)
        total_interactions_query = "SELECT SUM(total) FROM user_history"
        total_interactions = conn.execute(total_interactions_query).fetchone()[0]
        print(f"Total Recorded Interactions: {total_interactions:,}")

        # 3. In thử 3 dòng đầu tiên để kiểm tra cấu trúc
        # Thay vì in toàn bộ mảng (List), ta dùng hàm len() của DuckDB để xem kích thước mảng
        print("\nSample Data Structure (First 3 Users):")
        sample_query = """
            SELECT
                user_idx,
                user_id,
                total_abs_weight,
                total,
                seen_books,
                rated_books
            FROM user_history
            ORDER BY user_idx
            LIMIT 3
        """

        # Chuyển kết quả sang Pandas DataFrame để in ra màn hình cho đẹp
        sample_df = conn.execute(sample_query).fetchdf()
        print(sample_df.to_string(index=False))

    finally:
        conn.close()
        print("\nVerification complete. Connection closed.")

# =============================================================================
# Execution Block
# =============================================================================
if __name__ == "__main__":
    # Thay đổi đường dẫn này trỏ tới file user_history.db của bạn
    OUTPUT_DB_PATH = "/content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_history.db"

    verify_user_history_db(OUTPUT_DB_PATH)

Connecting to DuckDB database at: /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_history.db
Total Unique Users: 743,401
Total Recorded Interactions: 178,572,441

Sample Data Structure (First 3 Users):
 user_idx                          user_id  total_abs_weight  total                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import drive
import sys
import os
%pip install -q faiss-gpu-cu12
%pip install -q duckdb polars

sys.path.append(os.path.abspath('/content/drive/My Drive/Thesis/book_recsys'))
sys.path.insert(0,'/content/drive/My Drive/Thesis/book_recsys/app')

from app.recommenders.vector_recommender import VectorRecommender
from app.recommenders.processer import BookProcessor, UserProcessor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 21.7 MB/s eta 0:00:00


In [5]:
_base_dir = '/content/drive/My Drive/Thesis/book_recsys'
_data_dir = f'{_base_dir}/data/processed_2'
_models_dir = f'{_data_dir}/model_v1'
_eval_dir = f'{_data_dir}/eval_v1'

user_index = f'{_models_dir}/user_index.csv'
imtem_index = f'{_models_dir}/cb_sbert_book_index.csv'
history_db = f'{_models_dir}/user_history.db'

sbert_item_matrix = f'{_models_dir}/cb_sbert_matrix_v2.npy'
sbert_user_matrix = f'{_models_dir}/cb_sbert_user_matrix_v2_w1.npy'
sbert_item_faiss_index = f'{_models_dir}/cb_sbert_matrix_v2.index'

ials_user_matrix = f'{_models_dir}/cf_als_user_profiles.npy'
ials_item_matrix = f'{_models_dir}/cf_als_item_matrix.npy'
ials_item_faiss_index = f'{_models_dir}/cf_als_item_matrix.index'

test_file = f'{_data_dir}/test_main.csv'
ground_truth_file = f'{_eval_dir}/test_ground_truth.json'

cbf_predictions = f'{_eval_dir}/cbf_prediction.json'
cf_predictions = f'{_eval_dir}/cf_prediction.json'

In [6]:
import json
import time
import gc
# from processer import UserProcessor, BookProcessor
# from recommender import UniversalVectorRecommender

def get_chunks(lst, chunk_size=10_000):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

def append_predictions(predictions_dict, output_path, is_first_write=False):

    if not predictions_dict:
        return

    json_ready_data = {
        str(uid): [item.to_api_dict() for item in rec_items]
        for uid, rec_items in predictions_dict.items()
    }

    # Format từng key-value thành chuỗi chuẩn JSON
    items_str = []
    for k, v in json_ready_data.items():
        items_str.append(f'  "{k}": {json.dumps(v, ensure_ascii=False)}')

    with open(output_path, 'a', encoding='utf-8') as f:
        if not is_first_write:
            f.write(",\n")
        f.write(",\n".join(items_str))

def main():
    print("Start prediction")
    try:
      del cf_engine
      del cbf_engine
      del user_proc
      del book_proc
    except:
      pass
    gc.collect()
    # ==========================================
    # 1. Load user List
    # ==========================================
    gt_path = ground_truth_file
    with open(gt_path, 'r', encoding='utf-8') as f:
        ground_truth_data = json.load(f)
    test_users = list(ground_truth_data.keys())
    print(f"loading {len(test_users):,} users from Test.")

    # ==========================================
    # 2. Initializting
    # ==========================================
    print("-------Data Processors...")
    book_proc = BookProcessor(item_index_path= imtem_index)
    user_proc = UserProcessor(user_index_path= user_index, history_db_path= history_db)
    gc.collect()
    chunks = list(get_chunks(test_users, 5000))

    # ==========================================
    # 3.  CF (ALS)
    # ==========================================
    # print("\n" + "="*40 + "\nRunning CF (ALS)...")
    # cf_engine = VectorRecommender(
    #     book_processor= book_proc,
    #     user_processor= user_proc,
    #     faiss_index_path=ials_item_faiss_index,
    #     user_vectors_path = ials_user_matrix,
    # )


    # start_time = time.time()
    # cf_output_path = f"{_eval_dir}/cf_test_predictions.json"

    # Khởi tạo file JSON với dấu mở ngoặc
    # with open(cf_output_path, 'w', encoding='utf-8') as f:
    #     f.write("{\n")

    # is_first_write = True
    # for i, chunk in enumerate(chunks, 1):
    #     print(f" -> Processing {i}/{len(chunks)}...", end="\r")
    #     preds = cf_engine.recommend_batch(chunk, k=50)

    #     if preds:
    #         append_predictions(preds, cf_output_path, is_first_write)
    #         is_first_write = False

    # # Đóng file JSON
    # with open(cf_output_path, 'a', encoding='utf-8') as f:
    #     f.write("\n}\n")

    # print(f"\nCF Done {time.time() - start_time:.2f} second.")
    # del cf_engine
    # gc.collect()

    # dimension = 64
    # while dimension <=128:
    #   print(f" -> Processing {dimension}/{128}...", end="\r")
    #   cf_engine = VectorRecommender(
    #     book_processor= book_proc,
    #     user_processor= user_proc,
    #     faiss_index_path=f'{_data_dir}/model_v3/cf_als_item_matrix__{dimension}.index',
    #     user_vectors_path = f'{_data_dir}/model_v3/cf_als_user_profiles_{dimension}.npy',
    #   )

    #   start_time = time.time()
    #   cf_output_path = f"{_eval_dir}/cf_test_predictions_{dimension}.json"

    #   # Khởi tạo file JSON với dấu mở ngoặc
    #   with open(cf_output_path, 'w', encoding='utf-8') as f:
    #       f.write("{\n")

    #   is_first_write = True
    #   for i, chunk in enumerate(chunks, 1):
    #       print(f" -> Processing {i}/{len(chunks)}...", end="\r")
    #       preds = cf_engine.recommend_batch(chunk, k=50)

    #       if preds:
    #           append_predictions(preds, cf_output_path, is_first_write)
    #           is_first_write = False

    #   # Đóng file JSON
    #   with open(cf_output_path, 'a', encoding='utf-8') as f:
    #       f.write("\n}\n")

    #   print(f"\nCF Done {time.time() - start_time:.2f} second.")
    #   del cf_engine
    #   gc.collect()
    #   i=i*2

    # ==========================================
    # 4. CBF (NLP)
    # ==========================================
    print("="*40 + "\nRunning CBF (NLP)...")
    cbf_engine = VectorRecommender(
        book_processor= book_proc,
        user_processor= user_proc,
        faiss_index_path=sbert_item_faiss_index,
        user_vectors_path = sbert_user_matrix,
    )

    start_time = time.time()
    cbf_output_path = f"{_eval_dir}/cbf_test_predictions_v2_w1.json"

    # Khởi tạo file JSON với dấu mở ngoặc
    with open(cbf_output_path, 'w', encoding='utf-8') as f:
        f.write("{\n")

    is_first_write = True
    for i, chunk in enumerate(chunks, 1):
        print(f" ---- Processing {i}/{len(chunks)}...", end="\r")
        preds = cbf_engine.recommend_batch(chunk, k=50)

        if preds:
            append_predictions(preds, cbf_output_path, is_first_write)
            is_first_write = False

    # Đóng file JSON
    with open(cbf_output_path, 'a', encoding='utf-8') as f:
        f.write("\n}\n")

    print(f"\nDone {time.time() - start_time:.2f} giây.")
    print("ALL DONE")

if __name__ == "__main__":
    main()


Start prediction
loading 743,401 users from Test.
-------Data Processors...
Loading Book mapping from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/cb_sbert_book_index.csv ...
 -> Loaded 1,173,713 books into RAM dictionary.
Loading User mapping from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_index.csv...
 -> Loaded 743,401 users into RAM dictionary.
Building DuckDB connection pool (size=4)...
 -> Pool ready (4 connections to user_history.db)
Running CBF (NLP)...
Loading Data & CPU FAISS Index...
Cloning FAISS Index to GPU VRAM (for Batch Processing)...
Recommender Engine Ready (Users: 743,401)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Done 4166.00 giây.
ALL DONE


In [ ]:
import numpy as np
um = np.load(OUTPUT_MATRIX, mmap_mode='r')
print("Hoàn tất! Cấu trúc Matrix Output là:", um.shape)

### Kiểm tra lại thành phẩm sau khi Extract

In [ ]:
users_df = pd.read_csv(OUTPUT_INDEX)
print(f"Created Vectors for {len(users_df)} Unique Identifiers")
display(users_df.head())

import json
with open(OUTPUT_METADATA, 'r', encoding='utf-8') as f:
    user_metadata = json.load(f)

sample_uid = str(users_df.iloc[-1]['user_id'])
print(f"\nSample Metadata History for Random User: {sample_uid}")
print(json.dumps(user_metadata.get(sample_uid, {}), indent=4))

### Online Mode (CRUD)
Để Demo chức năng Cập Nhật / Thêm User Online cho API Server phục vụ Update History Profile.

In [ ]:
# Khai báo mảng rỗng cho User mới hoàn toàn
new_user_id = "TEST_NEW_USER_9999"
builder.add_new_user(user_id=new_user_id)

# Update Profile theo 1 cuốn sách đọc lướt qua (mặc định weight 0.5)
book_sample_id = "386162" # thay thế bằng book_id thật sự có trong list
builder.update_user_vector(user_id=new_user_id, book_id=book_sample_id, rating=None)

# Update Profile theo 1 cuốn sách thích mạnh (Rating = 5 sao)
builder.update_user_vector(user_id=new_user_id, book_id=book_sample_id, rating=5.0)

# In lại kết quả lịch sử Metadata Update để chứng thực
print(json.dumps(builder.user_history[new_user_id], indent=4))